In [1]:
# 1. Cài đặt thư viện cần thiết (nếu thiếu)

import subprocess
import sys

# Cài gymnasium (thường chưa có sẵn trên Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

print("Thư viện đã sẵn sàng.")

Thư viện đã sẵn sàng.


## 2. Clone GitHub Repo vào `/kaggle/working/repo`

Sử dụng Kaggle Secret `GH_TOKEN` để clone repo chứa mã nguồn PPO + LSTM.

In [2]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import sys

# =============================================================
# CẬP NHẬT TÊN REPO TẠI ĐÂY
# =============================================================
GITHUB_REPO  = "sin0235/Project" 

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GH_TOKEN")
BRANCH       = "master"


import subprocess, sys
from pathlib import Path

CLONE_DIR   = Path("/kaggle/working/repo")
WORKING_DIR = Path("/kaggle/working")

if CLONE_DIR.exists():
    result = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
else:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH, repo_url, str(CLONE_DIR)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Git clone/pull thất bại. Kiểm tra lại TOKEN và REPO.")

PROJECT_ROOT = CLONE_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Cloning into '/kaggle/working/repo'...

Project root: /kaggle/working/repo


## 3. Đọc cấu hình PPO từ `Conf/ppo_conf.yaml`

File config nằm trong repo vừa clone, dùng để điều chỉnh nhanh hyperparameter khi train trên Kaggle.

In [3]:
import yaml
import numpy as np
from pprint import pprint

CONF_PATH = PROJECT_ROOT / "Conf" / "ppo_conf.yaml"

with open(CONF_PATH, "r", encoding="utf-8") as f:
    ppo_cfg = yaml.safe_load(f)

# -------------------------------------------------------------
# OVERRIDE đường dẫn data khi chạy trên Kaggle
# -------------------------------------------------------------
# Dataset Kaggle: /kaggle/input/datasets/phctontrn/dataset-trading-rl
# Giả định các file ACB.csv, BID.csv,... nằm trực tiếp trong thư mục này.
ppo_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

print("Đường dẫn config:", CONF_PATH)
print("Data path (Kaggle):", ppo_cfg["data_path"])
print("\nCấu hình PPO (rút gọn):")
keys_to_show = [
    "tickers", "features", "window_size",
    "train_ratio", "val_ratio", "test_ratio",
    "initial_balance", "fee_rate", "reward_name", "reward_window",
    "learning_rate", "n_steps", "batch_size", "total_timesteps",
]
summary = {k: ppo_cfg.get(k) for k in keys_to_show}
pprint(summary)

Đường dẫn config: /kaggle/working/repo/Conf/ppo_conf.yaml
Data path (Kaggle): /kaggle/input/datasets/phctontrn/dataset-trading-rl

Cấu hình PPO (rút gọn):
{'batch_size': 256,
 'features': ['close_norm',
              'return_1d',
              'return_5d',
              'macd',
              'rsi',
              'adx',
              'volume_norm'],
 'fee_rate': 0.001,
 'initial_balance': 1000000000,
 'learning_rate': 0.00015,
 'n_steps': 1024,
 'reward_name': 'tmp',
 'reward_window': 20,
 'test_ratio': 0.15,
 'tickers': ['ACB',
             'BID',
             'BVH',
             'CTG',
             'FPT',
             'GAS',
             'HPG',
             'MBB',
             'MSN',
             'MWG',
             'SHB',
             'SSI',
             'STB',
             'VCB',
             'VIC',
             'VNM'],
 'total_timesteps': 200000,
 'train_ratio': 0.7,
 'val_ratio': 0.15,
 'window_size': 30}


## 4. Chạy training PPO + LSTM

Cell dưới sẽ gọi trực tiếp `train_ppo(config)` trong `src/training/PPO.py`.

- Logger sẽ tự tạo `run_id` và ghi log vào `results/runs/<run_id>` bên trong repo
- Khi muốn thử hyperparameters khác, chỉ cần chỉnh `Conf/ppo_conf.yaml` rồi chạy lại cell này.

In [4]:
import os
import sys

# Đảm bảo PROJECT_ROOT đã nằm trong sys.path từ cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("CWD:", os.getcwd())

from src.training.PPO import train_ppo

# Có thể override thêm một số hyperparameters ngay tại notebook nếu muốn, ví dụ:
# ppo_cfg["total_timesteps"] = 200_000
# ppo_cfg["learning_rate"] = 2e-4

agent, test_metrics = train_ppo(ppo_cfg)

print("\nKết quả test trung bình (tóm tắt):")
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v}")

CWD: /kaggle/working/repo
Data split: {'train': {'start': '2015-01-05', 'end': '2022-09-16', 'days': 1917}, 'val': {'start': '2022-09-19', 'end': '2024-05-14', 'days': 411}, 'test': {'start': '2024-05-15', 'end': '2025-12-31', 'days': 412}}


2026-03-21 17:32:39 [INFO] Run bắt đầu: ppo_20260321_173239 | Agent: PPO_LSTM | 2026-03-21 17:32:39
2026-03-21 17:32:39 [INFO] Data split: {'train': {'start': '2015-01-05', 'end': '2022-09-16', 'days': 1917}, 'val': {'start': '2022-09-19', 'end': '2024-05-14', 'days': 411}, 'test': {'start': '2024-05-15', 'end': '2025-12-31', 'days': 412}}
2026-03-21 17:32:39 [INFO] Device: cuda | Params: 179,218
2026-03-21 17:32:39 [INFO] Reward function: tmp | reward_window=20 | reward_scaling=1.0
2026-03-21 17:32:39 [INFO] Execution filters: trade_deadband=0.025 | max_weight_change_per_step=0.100
2026-03-21 17:32:39 [INFO] LR schedule: linear | base_lr=1.50e-04 | min_lr=3.00e-05


Device: cuda
Model params: 179,218


2026-03-21 17:32:44 [INFO] [Ep      1] reward=-142.6814 | value=     954,754,826 | return= -4.52% | trades=2678 | cost=168,783,774
2026-03-21 17:32:44 [INFO] [Ep      2] reward=-136.0302 | value=   1,248,634,630 | return= 24.86% | trades=2726 | cost=191,016,680
2026-03-21 17:32:44 [INFO] [Ep      3] reward=-123.3205 | value=   1,028,538,203 | return=  2.85% | trades=2741 | cost=173,377,491
2026-03-21 17:32:44 [INFO] [Ep      4] reward=-203.7697 | value=     831,277,902 | return=-16.87% | trades=2705 | cost=179,690,855
2026-03-21 17:32:44 [INFO] [Rollout 1/196] steps=1,024 | pi_loss=-0.01123 | v_loss=167.31164 | kl=0.00187 | lr=1.50e-04
2026-03-21 17:32:46 [INFO] [Ep      5] reward=-174.1991 | value=     970,502,732 | return= -2.95% | trades=2676 | cost=191,013,864
2026-03-21 17:32:46 [INFO] [Ep      6] reward=-133.8577 | value=   1,271,822,323 | return= 27.18% | trades=2645 | cost=206,721,257
2026-03-21 17:32:46 [INFO] [Ep      7] reward=-151.6190 | value=     935,874,364 | return= -6.


Kết quả test trung bình (tóm tắt):
  annualized_return: 0.24825896221587151
  annualized_vol: 0.17436547051319606
  avg_daily_return: 0.0009407870576044646
  calmar_ratio: 1.4477828978989238
  cvar_95: 0.026214444179831382
  final_value: 1399541191.5680003
  initial_value: 1000000000.0
  kurtosis: 11.043473740096493
  max_drawdown: 0.1714752692383327
  max_drawdown_duration: 81.0
  profit_factor: 1.3093581033197206
  sharpe_ratio: 1.1015847228869118
  skewness: -0.6442197175424366
  sortino_ratio: 1.0476678801251906
  std_daily_return: 0.010969605821115721
  total_return: 0.39954119156800033
  var_95: 0.012379775982209728
  win_rate: 0.5497382198952879


## 5. Eval chi tiết và xem lại kết quả

Cell dưới cho phép bạn **chạy lại eval** (không train thêm) từ checkpoint phù hợp nhất của run gần nhất: ưu tiên `best_model.pt`, nếu thiếu thì fallback sang `final_model.pt` hoặc `checkpoint_*.pt`, rồi in full metrics để tiện so sánh giữa các run.

In [5]:
import sys
from pathlib import Path
import numpy as np

# Đảm bảo PROJECT_ROOT đã được set ở cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import (
    get_results_root_candidates,
    resolve_eval_run_across_roots,
    resolve_ppo_config,
)
from src.utils.metrics import compute_all, format_report
from src.environment.trading_env import TradingEnv
from src.utils.data_splitter import load_data, split_by_ratio

# Eval lại: tự tìm run mới nhất còn dùng được.
# Ưu tiên config.json của run; nếu thiếu sẽ cố suy luận shape model từ checkpoint.

from src.models.lstm import PPOLSTMActorCritic
from src.agents.ppo_agent import PPOAgent

base_cfg = resolve_ppo_config()
base_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

candidate_roots = get_results_root_candidates(project_root=PROJECT_ROOT)
resolved = resolve_eval_run_across_roots(
    candidate_roots,
    base_config=base_cfg,
    overrides={
        "data_path": "/kaggle/input/datasets/phctontrn/dataset-trading-rl",
        "device": "auto",
    },
)
if resolved["run_dir"] is None:
    print("Không tìm thấy run nào đủ artifact để eval.")
    if resolved["checked_results_roots"]:
        print("Đã kiểm tra các thư mục:")
        for root in resolved["checked_results_roots"]:
            print(" -", root)
    if resolved["missing_results_roots"]:
        print("Thiếu thư mục:")
        for root in resolved["missing_results_roots"]:
            print(" -", root)
    if resolved["all_skipped_runs"]:
        print("Các run bị bỏ qua:")
        for item in resolved["all_skipped_runs"]:
            print(" -", item["run_id"], "->", item["reason"], "| root:", item["results_root"])
else:
        latest_run = resolved["run_dir"]
        ckpt_path = resolved["ckpt_path"]
        ckpt_source = resolved["ckpt_source"]
        cfg = resolved["config"]
        print("Results root:", resolved["results_root"])
        print("Latest evaluable run:", latest_run.name)
        print("Config source:", resolved["config_source"])
        print("Checkpoint source:", ckpt_source)
        if resolved["all_skipped_runs"]:
            print("Skipped runs during search:")
            for item in resolved["all_skipped_runs"]:
                print(" -", item["run_id"], "->", item["reason"], "| root:", item["results_root"])

        data_dict = load_data(tickers=cfg["tickers"], data_path=cfg["data_path"])
        split = split_by_ratio(
            data_dict,
            train_ratio=cfg["train_ratio"],
            val_ratio=cfg["val_ratio"],
            test_ratio=cfg["test_ratio"],
        )
        print("Data split summary:", split.summary())

        test_env = TradingEnv(
            tickers=cfg["tickers"],
            mode="continuous",
            initial_balance=cfg["initial_balance"],
            fee_rate=cfg["fee_rate"],
            window_size=cfg["window_size"],
            data_dict=split.test,
            features=cfg["features"],
            max_steps=cfg["max_steps_eval"],
            random_start=False,
            reward_scaling=cfg["reward_scaling"],
            reward_name=cfg["reward_name"],
            reward_kwargs={"window": cfg["reward_window"]},
            trade_deadband=cfg["trade_deadband"],
            max_weight_change_per_step=cfg["max_weight_change_per_step"],
            print_verbosity=999999,
        )

        state_space = test_env.state_space
        n_stocks = state_space.n_stocks
        n_features = state_space.n_features

        model = PPOLSTMActorCritic(
            n_stocks=n_stocks,
            n_features=n_features,
            seq_len=cfg["window_size"],
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
            log_std_init=cfg["log_std_init"],
        )

        agent = PPOAgent(
            model=model,
            lr=cfg["learning_rate"],
            gamma=cfg["gamma"],
            gae_lambda=cfg["gae_lambda"],
            clip_range=cfg["clip_range"],
            ent_coef=cfg["ent_coef"],
            vf_coef=cfg["vf_coef"],
            max_grad_norm=cfg["max_grad_norm"],
            target_kl=cfg["target_kl"],
            n_epochs=cfg["n_epochs"],
            batch_size=cfg["batch_size"],
            device=cfg["device"],
        )

        print(f"Đang load checkpoint ({ckpt_source}):", ckpt_path)
        agent.load(str(ckpt_path))

        # Đánh giá trên test set
        print("\nĐang eval trên test set...")
        values_list = agent.evaluate(test_env, state_space, n_episodes=cfg["n_eval_episodes"])
        metrics_list = [compute_all(v, cfg["initial_balance"]) for v in values_list]

        avg_metrics = {}
        for key in metrics_list[0]:
            vals = [m[key] for m in metrics_list if key in m]
            avg_metrics[key] = float(np.mean(vals))

        print("\n=== TEST METRICS (TRUNG BÌNH) ===")
        print(format_report(avg_metrics))

Results root: /kaggle/working/repo/results/runs
Latest evaluable run: ppo_20260321_173239
Config source: run_config
Checkpoint source: best_model
Data split summary: {'train': {'start': '2015-01-05', 'end': '2022-09-16', 'days': 1917}, 'val': {'start': '2022-09-19', 'end': '2024-05-14', 'days': 411}, 'test': {'start': '2024-05-15', 'end': '2025-12-31', 'days': 412}}
Đang load checkpoint (best_model): /kaggle/working/repo/results/runs/ppo_20260321_173239/checkpoints/best_model.pt

Đang eval trên test set...

=== TEST METRICS (TRUNG BÌNH) ===
                  PERFORMANCE REPORT                   
  Vốn ban đầu          :      1,000,000,000 VND
  Giá trị cuối kỳ      :      1,399,541,192 VND
-------------------------------------------------------
  Tổng lợi nhuận       :            39.95%
  Lợi nhuận hàng năm   :            24.83%
  Biến động hàng năm   :            17.44%
-------------------------------------------------------
  Sharpe Ratio         :            1.1016
  Sortino Ratio  

In [6]:
import shutil
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import get_results_root_candidates

# Zip toàn bộ thư mục runs để tải/lưu trữ nhanh
candidate_roots = get_results_root_candidates(project_root=PROJECT_ROOT)
runs_dir = next((root for root in candidate_roots if root.exists() and any(root.iterdir())), None)

if runs_dir is not None:
    zip_base = runs_dir.parent / "runs_backup"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=runs_dir)
    print(f"Da tao file zip: {zip_path}")
    print(f"Source runs dir: {runs_dir}")
else:
    print("Khong tim thay thu muc runs nao co du lieu.")
    print("Da kiem tra:")
    for root in candidate_roots:
        print(" -", root)

Da tao file zip: /kaggle/working/repo/results/runs_backup.zip
Source runs dir: /kaggle/working/repo/results/runs
